# 09 — Reranker kiểu HySAR (LightGBM) trên ứng viên của kNN và GRU4Rec

Hướng thực dụng cho đa hành vi theo tài liệu tổng quan phương pháp (HySAR, UMUAI 2023): một mô hình xếp hạng học từ đặc trưng hành vi sắp xếp lại ứng viên của các mô hình gốc.

- Ứng viên của query = hợp của top-100 từ **mô hình kNN/SR tốt nhất** (notebook 06), top-100 từ **GRU4Rec** (notebook 07) và các item trong prefix.
- Đặc trưng (`rr.rerank`): điểm và hạng của từng mô hình gốc, số lần view/addtocart/transaction của item trong prefix, độ gần đây, item có ở nhóm cuối không, độ phổ biến theo hành vi trong train, độ dài prefix, item có cùng danh mục (as-of `t`) với nhóm cuối không.
- Nhãn: item nằm trong target của bài toán 1. LightGBM `lambdarank`.
- Dữ liệu học: query **validation**, mô hình gốc fit trên train. Siêu tham số chọn bằng kiểm định chéo 2 fold **theo phiên** trên validation; kết quả validation báo cáo là **out-of-fold** (mỗi query được chấm bởi mô hình không thấy phiên của nó). Mô hình cuối học trên toàn bộ validation.
- Đầu ra: `models/reranker/{model.txt, config.json}`, `reports/eval/reranker_validation.json`. **Không đọc test.**

## 1. Đường dẫn, giao thức và kiểm tra đầu vào

In [ ]:
from pathlib import Path
import sys, time

ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "src/rr").is_dir())
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import torch
import lightgbm as lgb
from IPython.display import display
from rr.io import environment, fingerprint, read_json, rel, require, utc_now, verify_inputs, write_json
from rr.candidates import CandidateFilter
from rr.samples import prefix_events
from rr.models import popularity, prefix_matrix
from rr.evaluation import load_samples, run, flat, fit_neighbour_model, recommend_neighbour
from rr.gru import GroupData, build_from_config, idx_to_itemid, query_states, topn_items
from rr.rerank import add_features, candidate_table, feature_columns, label, ranked_lists, session_folds

PROTOCOL_PATH = ROOT / "configs/protocol.json"
PROTOCOL = read_json(PROTOCOL_PATH)
PROCESSED = ROOT / "data/processed"
EVAL_DIR = ROOT / "reports/eval"
MODEL_DIR = ROOT / "models/reranker"
FALLBACK_TOP_N = PROTOCOL["candidates"]["fallback_top_n"]
MAX_GROUPS = PROTOCOL["knn"]["max_prefix_groups"]
SEED = PROTOCOL["seed"]
TOP_N, WIDTH = 100, 200
torch.set_num_threads(4)

samples_report = read_json(ROOT / "reports/samples/metrics.json")
metadata_report = read_json(ROOT / "reports/metadata/metrics.json")
split_report = read_json(ROOT / "reports/session_split/metrics.json")
pop_report = read_json(EVAL_DIR / "popularity_validation.json")
knn_report = read_json(EVAL_DIR / "knn_validation.json")
gru_report = read_json(EVAL_DIR / "gru4rec_validation.json")
for name, r in [("samples", samples_report), ("popularity", pop_report), ("knn", knn_report), ("gru4rec", gru_report)]:
    require(r["protocol"]["version"] == PROTOCOL["version"], f"{name} được tạo với protocol khác.")
expected = {p: samples_report["artifacts"][p] for p in [
    "data/processed/item_vocab.parquet", "data/processed/samples/validation_queries.parquet",
    "data/processed/samples/validation_targets.parquet"]}
for p in ["data/processed/item_property_history.parquet", "data/processed/item_catalog.parquet"]:
    expected[p] = metadata_report["artifacts"][p]
expected["data/processed/events_sessionized.parquet"] = split_report["artifacts"]["data/processed/events_sessionized.parquet"]
expected[gru_report["model_file"]["path"]] = gru_report["model_file"]
inputs = verify_inputs(ROOT, {k: {"bytes": v["bytes"], "sha256": v["sha256"]} for k, v in expected.items()})
print("Đầu vào khớp SHA-256:", *inputs, sep="\n  ")
print(environment(), "| lightgbm", lgb.__version__, "| protocol", PROTOCOL["version"])

## 2. Ví dụ biên

Hợp ứng viên giữ hạng của từng nguồn (thiếu → `TOP_N + 1`), item trong prefix luôn là ứng viên, nhãn chỉ bật cho item trong target.

In [ ]:
_q = pd.DataFrame({"query_id": [1], "session_id": [9], "prefix_len_groups": [1], "query_time_ms": [10]})
_p = pd.DataFrame({"query_id": [1], "itemid": [5], "event": ["addtocart"], "distance": [0]})
_t = candidate_table([1], {"a": (np.array([[7, 8]]), np.array([[1.0, 0.5]]))}, _p, top_n=2)
assert _t["itemid"].tolist() == [5, 7, 8] and _t["a_rank"].tolist() == [3, 1, 2]
assert label(_t, pd.DataFrame({"query_id": [1], "itemid": [8]})).tolist() == [0, 0, 1]
_f = add_features(_t, _q, _p, pd.DataFrame({"itemid": [7], "event": ["view"]}),
                  pd.DataFrame(columns=["itemid", "timestamp", "property", "value"]).astype({"itemid": "int64", "timestamp": "int64"}))
assert _f.set_index("itemid").loc[5, "prefix_addtocart"] == 1
del _q, _p, _t, _f
print("Đã kiểm tra hợp ứng viên, hạng thiếu và nhãn.")

## 3. Ứng viên từ các mô hình gốc (fit trên train)

In [ ]:
vocab = pd.read_parquet(PROCESSED / "item_vocab.parquet")
catalog = pd.read_parquet(PROCESSED / "item_catalog.parquet")
history = pd.read_parquet(PROCESSED / "item_property_history.parquet")
events = pd.read_parquet(PROCESSED / "events_sessionized.parquet",
                         columns=["session_id", "group_index", "itemid", "event", "timestamp", "split"])
events["split"] = events["split"].astype(str)
train_events = events[events["split"] == "train"]
val_events = events[events["split"] == "validation"]
flt = CandidateFilter.from_frames(vocab, catalog, history, "known_at_train_ms")
fb_items, fb_scores = (a[:FALLBACK_TOP_N] for a in popularity(train_events, pop_report["selected"]["fallback_weights"]))
val_q, val_t = load_samples(ROOT, "validation")
qids, qtimes = val_q["query_id"].to_numpy(), val_q["query_time_ms"].to_numpy()
prefix = prefix_events(val_q, val_events, ("itemid", "event"), max_groups=MAX_GROUPS)

selected = knn_report["selected"]
knn_family = max(selected, key=lambda f: knn_report["runs"][selected[f]["best_run"]]["all"]["ndcg@20"])
t0 = time.perf_counter()
knn_model = fit_neighbour_model(knn_family, selected, train_events)
knn_lists = recommend_neighbour(knn_model, knn_family, selected, prefix, qids, qtimes, WIDTH)
gru_cfg = read_json(ROOT / "models/gru4rec/config.json")
gru = build_from_config(gru_cfg)
gru.load_state_dict(torch.load(ROOT / "models/gru4rec/model.pt"))
gru_lists = topn_items(gru, query_states(gru, GroupData(val_events, vocab), val_q), idx_to_itemid(vocab), WIDTH)
print(f"kNN tốt nhất: {knn_family} ({selected[knn_family]['best_run']}) | GRU4Rec: {gru_cfg['name']} | {time.perf_counter() - t0:.0f}s")
base_reports = {f"{knn_family}": run(*knn_lists, val_q, val_t, flt, fb_items, fb_scores)[0],
                "gru4rec": run(*gru_lists, val_q, val_t, flt, fb_items, fb_scores)[0]}
require(abs(base_reports[knn_family]["all"]["ndcg@20"] - knn_report["runs"][selected[knn_family]["best_run"]]["all"]["ndcg@20"]) < 1e-9,
        "Fit lại kNN không khớp kết quả notebook 06.")

## 4. Bảng ứng viên, đặc trưng và độ phủ

Độ phủ = tỷ lệ lượt target nằm trong tập ứng viên; đây là trần Recall của reranker trước bước lấp fallback.

In [ ]:
sources = {"knn": knn_lists, "gru": gru_lists}
t0 = time.perf_counter()
table = candidate_table(qids, sources, prefix, top_n=TOP_N)
table = add_features(table, val_q, prefix, train_events, history)
table["y"] = label(table, val_t)
FEATURES = feature_columns(table, sources)
coverage = float(table["y"].sum() / len(val_t))
print(f"{len(table):,} cặp (query, item) | {len(table) / len(val_q):.0f} ứng viên/query | độ phủ target {coverage:.2%} | {time.perf_counter() - t0:.0f}s")
display(table.head())

## 5. Kiểm định chéo 2 fold theo phiên và chọn siêu tham số

Mỗi fold: học trên query (có ít nhất một nhãn dương) của fold kia, dừng sớm theo NDCG@20 của LightGBM trên fold được giữ lại, rồi chấm điểm fold được giữ lại. Chọn cấu hình theo NDCG@20 **của bộ đánh giá chung** trên dự đoán out-of-fold.

In [ ]:
folds = session_folds(val_q, 2, seed=SEED)
fold_of_row = pd.Series(folds, index=qids).reindex(table["query_id"].to_numpy()).to_numpy()
has_pos = table.groupby("query_id")["y"].transform("max").to_numpy() > 0

def lgb_data(mask):
    part = table[mask]
    return lgb.Dataset(part[FEATURES], part["y"], group=part.groupby("query_id", sort=False).size().to_numpy(),
                       free_raw_data=False)

GRID = [{"num_leaves": 31, "min_data_in_leaf": 100, "learning_rate": 0.05},
        {"num_leaves": 63, "min_data_in_leaf": 50, "learning_rate": 0.05}]
cv = {}
for gi, params in enumerate(GRID):
    full = {"objective": "lambdarank", "metric": "ndcg", "eval_at": [20], "lambdarank_truncation_level": 20,
            "feature_fraction": 0.9, "bagging_fraction": 0.8, "bagging_freq": 1, "seed": SEED,
            "deterministic": True, "num_threads": 4, "verbose": -1, **params}
    oof = np.zeros(len(table))
    best_iters = []
    for k in [0, 1]:
        tr, va = (fold_of_row != k) & has_pos, (fold_of_row == k) & has_pos
        booster = lgb.train(full, lgb_data(tr), num_boost_round=1000, valid_sets=[lgb_data(va)],
                            callbacks=[lgb.early_stopping(50, verbose=False)])
        best_iters.append(booster.best_iteration)
        held = fold_of_row == k
        oof[held] = booster.predict(table.loc[held, FEATURES], num_iteration=booster.best_iteration)
    items, scores = ranked_lists(table, oof, qids, WIDTH)
    report, per_q, *_ = run(items, scores, val_q, val_t, flt, fb_items, fb_scores)
    cv[f"grid{gi}"] = {"params": full, "best_iterations": best_iters, "report": report}
    print(f"grid{gi} {params}: OOF ndcg@20={report['all']['ndcg@20']:.4f} iters={best_iters}")
best_cv = max(cv, key=lambda n: cv[n]["report"]["all"]["ndcg@20"])

## 6. So sánh với mô hình gốc trên cùng bộ query validation

Reranker dùng kết quả out-of-fold; mô hình gốc dùng đúng danh sách của chúng. Cả hai bài toán, tách target mới/lặp.

In [ ]:
compare = {**base_reports, "reranker (OOF)": cv[best_cv]["report"]}
for part in ["all", "new_targets", "repeat_targets", "strong_behavior_targets"]:
    print(part)
    display(flat(compare, part)[["hit@20", "recall@20", "mrr@20", "ndcg@20"]].style.format("{:.4f}"))

## 7. Mô hình cuối trên toàn bộ validation và lưu

In [ ]:
final_params = cv[best_cv]["params"]
n_rounds = int(np.mean(cv[best_cv]["best_iterations"]))
final = lgb.train(final_params, lgb_data(has_pos), num_boost_round=n_rounds)
importance = pd.Series(final.feature_importance("gain"), index=FEATURES).sort_values(ascending=False)
display(importance.to_frame("gain"))
MODEL_DIR.mkdir(parents=True, exist_ok=True)
# LightGBM's C++ writer cannot open paths with non-ASCII characters; write the model string from Python.
(MODEL_DIR / "model.txt").write_bytes(final.model_to_string().encode("utf-8"))
config = {"features": FEATURES, "sources": {"knn": knn_family, "gru": gru_cfg["name"]}, "top_n": TOP_N,
          "params": final_params, "num_boost_round": n_rounds, "trained_on": "validation queries (base models fitted on train)"}
write_json(MODEL_DIR / "config.json", config)
require(all(fingerprint(ROOT / p) == v for p, v in inputs.items()), "Đầu vào thay đổi trong lúc chạy.")
write_json(EVAL_DIR / "reranker_validation.json", {
    "generated_at_utc": utc_now(), **environment(), "lightgbm_version": lgb.__version__, "notebook": "notebooks/09_reranker_hysar.ipynb",
    "protocol": {"path": rel(PROTOCOL_PATH, ROOT), "version": PROTOCOL["version"], **fingerprint(PROTOCOL_PATH)},
    "inputs": inputs, "split": "validation", "n_queries": len(val_q), "candidate_pairs": len(table),
    "candidate_target_coverage": coverage, "base_models": base_reports, "cv": cv, "selected": best_cv, "config": config,
    "feature_importance_gain": importance.to_dict(),
    "model_file": {"path": rel(MODEL_DIR / "model.txt", ROOT), **fingerprint(MODEL_DIR / "model.txt")}, "status": "PASS"})
print("Báo cáo:", rel(EVAL_DIR / "reranker_validation.json", ROOT))